# Design an entire CDR3

`predict_cdr3.ipynb` masks a known CDR3 and inspects the top-5 candidates *per position*. This notebook does the harder thing: mask the **whole loop at once** and read off one complete designed CDR-H3, given only the rest of the antibody and the antigen.

That is the task setting structure-based antibody design methods are benchmarked on -- redesign the CDR-H3 loop given the complex. The difference is that those methods condition on the antibody-antigen complex's 3D structure, and LangAAI never sees structure at all: only two sequences.

In [ ]:
import langaai

model = langaai.load()
print(model.device, model.dim)

A Fab against the SARS-CoV-2 receptor-binding domain (PDB `7wp8`). Variable domains only -- the model expects variable-domain sequences, not full chains with the constant region.

The CDR-H3 span comes from IMGT numbering of this antibody. Spans are *always* caller-supplied: this package bundles no CDR-numbering tool and cannot work out where CDR-H3 starts from a raw sequence.

In [ ]:
heavy = "QVQLQQPGAELVRPGASVKLSCKASGYTFTSYWMNWVKQRPEQGLEWIGRIDPYDSETHYNQKFKDKAILTVDKSSTTAYMQLSSLTSEDSAVYYCARWGTVEWFFDYWGQGTTLTVSQ"
light = "DIVMTQSPSSLAMSVGQKVTMSCKSSQSLLNSYNQENYLAWYQQKPGQSPKLLVYFASTRESGVPDRFIGSGSGTDFTLTISSVQAEDLADYFCQQHYSTPFTFGSGTKLEIK"
antigen = "CPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGTIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYRYRLFRKSNLKPFERDISTEIYQAGSKPCNGVKGFNCYFPLQSYGFQPTYGVGYQPYRVVVLSFELL"

span = (96, 108)                      # CDR-H3, half-open, 0-indexed into `heavy`
native = heavy[span[0]:span[1]]
print(f"native CDR-H3: {native} ({len(native)} residues)")

ab = model.encode_antibody(heavy, light)
ag = model.embed_antigen(antigen)
print(f"antibody: {len(ab)} tokens, antigen: {len(ag)} residues (truncated={ag.truncated})")

Mask the **entire** span. This is deliberately harsher than the model's training-time masking, which corrupts ~40% of CDR positions and leaves the rest visible: here nothing inside the loop is given away, so every residue has to come from the framework plus the antigen.

In [ ]:
masked = ab.mask_region("cdr3", spans=[span])
print(f"{len(masked.positions)} masked positions: {masked.positions}")

[predictions] = model.predict_masked([(masked, ag)], top_k=5)
designed = "".join(p.top[0][0] for p in predictions)

print(f"native:   {native}")
print(f"designed: {designed}")

Amino-acid recovery (AAR) is just the fraction of positions where the design matches the native loop -- the metric CDR-H3 design methods report. It is a *recovery* number, not a quality number: a design that recovers nothing may still be a fine binder, and a high-recovery design is not automatically one.

In [ ]:
def aar(pred: str, true: str) -> float:
    """Amino-acid recovery: fraction of matching residues."""
    return sum(p == t for p, t in zip(pred, true)) / len(true)


print(f"AAR = {aar(designed, native):.3f}  ({sum(p == t for p, t in zip(designed, native))}/{len(native)} residues)")

Per position, with the model's confidence. The pattern is the one to expect from a sequence-only model: the conserved anchors at each end of the loop come back almost exactly, and the variable middle -- the part that actually makes contact -- is where the design diverges.

In [ ]:
print(f"{'pos':>4}  {'native':^6}  {'designed':^8}  {'p':>5}   top-5")
hits, misses = [], []
for p, true_letter in zip(predictions, native):
    letter, prob = p.top[0]
    (hits if letter == true_letter else misses).append(prob)
    mark = "match" if letter == true_letter else ""
    top5 = " ".join(f"{a}:{v:.2f}" for a, v in p.top)
    print(f"{p.position:>4}  {true_letter:^6}  {letter:^8}  {prob:>5.2f}   {top5}  {mark}")

print()
print(f"mean confidence where the design matched:      {sum(hits) / len(hits):.3f}  (n={len(hits)})")
print(f"mean confidence where it did not:              {sum(misses) / len(misses):.3f}  (n={len(misses)})")

Two things this recipe is *not*:

- **Not a joint sample.** All positions are masked in one forward pass, so the model's distribution over the loop factorises: taking the arg-max at each position independently can produce a combination the model would never rank first as a whole sequence. `candidate_cdr3_search.ipynb` searches that space properly and shows how much the ranking moves once positions are allowed to condition on each other.
- **Not an affinity claim.** Recovering the native loop says the model has learned what belongs there; it says nothing about the binding affinity of any loop it writes instead.

For the full evaluation across every complex in the held-out split, and the comparison against structure-based methods, see the paper.